In [5]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

paths = {}
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.endswith('.json'):
            paths[filename] = os.path.join(dirname, filename)

print(paths)

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

{'yelp_academic_dataset_review.json': '/kaggle/input/yelp-dataset/yelp_academic_dataset_review.json', 'yelp_academic_dataset_checkin.json': '/kaggle/input/yelp-dataset/yelp_academic_dataset_checkin.json', 'yelp_academic_dataset_business.json': '/kaggle/input/yelp-dataset/yelp_academic_dataset_business.json', 'yelp_academic_dataset_tip.json': '/kaggle/input/yelp-dataset/yelp_academic_dataset_tip.json', 'yelp_academic_dataset_user.json': '/kaggle/input/yelp-dataset/yelp_academic_dataset_user.json'}


In [6]:
filtered_paths = {key.replace('yelp_', '').replace('.json', ''): value 
                  for key, value in paths.items()}

print(filtered_paths)


{'academic_dataset_review': '/kaggle/input/yelp-dataset/yelp_academic_dataset_review.json', 'academic_dataset_checkin': '/kaggle/input/yelp-dataset/yelp_academic_dataset_checkin.json', 'academic_dataset_business': '/kaggle/input/yelp-dataset/yelp_academic_dataset_business.json', 'academic_dataset_tip': '/kaggle/input/yelp-dataset/yelp_academic_dataset_tip.json', 'academic_dataset_user': '/kaggle/input/yelp-dataset/yelp_academic_dataset_user.json'}


In [7]:
# paths = [os.path.join(dirname, filename)
#          for dirname, _, filenames in os.walk('/kaggle/input')
#          for filename in filenames
#          if filename.endswith('.json')]

# print(paths)


In [8]:
from pyspark.sql import SparkSession
import requests
import json

# Create a Spark session
spark = SparkSession.builder \
    .appName("read jsonm") \
    .getOrCreate()
spark

In [13]:
import sqlite3
import pandas as pd

# Create a SQLite DB in Kaggle's working directory
conn = sqlite3.connect("yelp.db")
cursor = conn.cursor()

spark_dataframes = {}  # Dictionary to store Spark DataFrames
pandas_dataframes = {}  # Dictionary to store Pandas DataFrames

for key, value in filtered_paths.items():
    print(f"Processing DataFrame: {key}")
    spark_dataframes[key] = spark.read.json(value, multiLine=True)
    
    # Convert to Pandas DataFrame
    pandas_dataframes[key] = spark_dataframes[key].toPandas()
    
    # Flatten nested structures and clean data
    pandas_dataframes[key] = pandas_dataframes[key].applymap(
        lambda x: str(x) if isinstance(x, (list, dict)) else x
    )
    
    for col in pandas_dataframes[key].columns:
        if pandas_dataframes[key][col].dtype == 'object':
            pandas_dataframes[key][col] = pandas_dataframes[key][col].astype(str)
    
    # Save to SQLite database
    table_name = f"tbl_{key}"
    pandas_dataframes[key].to_sql(table_name, conn, if_exists='replace', index=False)
    print(f"Table '{table_name}' has been written to SQLite.")
# _dataframes[key].show(5)  # Display the first 5 rows of each DataFrame

# Close the connection
conn.close()


Processing DataFrame: academic_dataset_review


<ipython-input-13-2ab9cf86ad0b>:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pandas_dataframes[key] = pandas_dataframes[key].applymap(


Table 'tbl_academic_dataset_review' has been written to SQLite.
Processing DataFrame: academic_dataset_checkin
Table 'tbl_academic_dataset_checkin' has been written to SQLite.
Processing DataFrame: academic_dataset_business


<ipython-input-13-2ab9cf86ad0b>:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pandas_dataframes[key] = pandas_dataframes[key].applymap(
<ipython-input-13-2ab9cf86ad0b>:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pandas_dataframes[key] = pandas_dataframes[key].applymap(


Table 'tbl_academic_dataset_business' has been written to SQLite.
Processing DataFrame: academic_dataset_tip


<ipython-input-13-2ab9cf86ad0b>:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pandas_dataframes[key] = pandas_dataframes[key].applymap(


Table 'tbl_academic_dataset_tip' has been written to SQLite.
Processing DataFrame: academic_dataset_user
Table 'tbl_academic_dataset_user' has been written to SQLite.


<ipython-input-13-2ab9cf86ad0b>:19: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  pandas_dataframes[key] = pandas_dataframes[key].applymap(


In [14]:
spark.stop()
